# OpenMuCF quickstart

**Open FAIR rate ledger + differentiable muon-catalyzed-fusion cycle/energy auditor.**

This notebook walks the v1 spine end to end. Every number is produced by the engine from the
provenance-enforced rate ledger — nothing is hard-coded here. Positioning is honest: OpenMuCF is
*open, reproducible, differentiable, UQ-bearing infrastructure*, **not new physics**.

Install first (from the repo root):
```bash
python -m venv .venv && . .venv/bin/activate
pip install -e ".[dev]"
```

## 1. Load the validated FAIR rate ledger
The single source of truth: every rate carries provenance, conditions, an uncertainty band, an
established/contested tag, and a validity range.

In [ ]:
from openmucf import analytic, cycle, interop, load_rates
from openmucf.energy import EnergyChain

rates = load_rates()
print(f"{len(rates.symbols())} rates "
      f"({len(rates.contested())} contested, {len(rates.needs_verification())} need verification)")

## 2. Fusions per muon, from the differentiable cycle ODE
`fusions_per_muon_from_conditions` assembles the cycle rates from the ledger + physical conditions
`(T, phi, c_t)` and integrates the stiff diffrax network. At liquid-D/T-ish conditions this lands
near the historical record (~150).

In [ ]:
T, phi, c_t = 300.0, 1.2, 0.5
x_mu_ode = float(cycle.fusions_per_muon_from_conditions(rates, T=T, phi=phi, c_t=c_t))
print(f"X_mu(T={T} K, phi={phi}, c_t={c_t}) = {x_mu_ode:.1f}   # full diffrax ODE")

## 3. Closed-form cross-check (gate V1)
The analytic yield `X_mu = 1 / (omega_s^eff + lambda_0 / lambda_c)` is the regression backbone; the
ODE must reproduce it to < 1%.

In [ ]:
os0 = rates.value("omega_s0") / 100.0
ose = analytic.effective_sticking(os0, rates.value("R_col"))
lam_c = analytic.cycling_rate(phi, 1.2e8)
x_mu_analytic = float(analytic.fusions_per_muon(ose, lam_c))
print(f"closed-form X_mu = {x_mu_analytic:.1f}   (omega_s^eff = {ose*100:.3f}%)")

## 4. The honest energy ladder
Two explicit layers so nothing hides: scientific gain (fusion energy / muon-beam energy) vs
net-electrical gain through the full efficiency chain. The net-electrical breakeven sits far above
the usually-quoted scientific one — that gap is the skeptic's point, quantified.

In [ ]:
chain = EnergyChain()
print(f"record X_mu ~150 | sci breakeven {chain.breakeven_xmu_sci():.0f} "
      f"| net-electrical breakeven {chain.breakeven_xmu_net():.0f}")
print(f"Q_sci(150) = {chain.Q_sci(150):.3f}   Q_net(150) = {chain.Q_net_electrical(150):.3f}")

## 5. Export GEANT4-ready rate tables (interop)
The interop layer exports the differentiable rates as plain CSV/JSON tables (and offers a callable
API). It **complements** GEANT4 — OpenMuCF is the rate/kinetics/UQ layer and never re-runs transport.

In [ ]:
import tempfile

outdir = tempfile.mkdtemp(prefix="openmucf_export_")
written = interop.export_all(rates, outdir, fmt="both")
print(f"exported {sorted(written)} to {outdir}")

api = interop.geant4_callables(rates)
print(f"callable omega_s_eff(phi=1.2, T=300) = {api['omega_s_eff'](1.2, 300.0):.5f}")

## Next steps
- `make findings` — reproduce the sensitivity ranking + breakeven falsification (`FINDINGS.md`).
- `make validate` — reproduce the pre-registered literature targets (`VALIDATION.md`).
- `make calibration` — Bayesian calibration + identifiability (`CALIBRATION.md`).
- See `docs/` for the API overview and the roadmap toward the Phase-3 sticking surrogate.